# Banking Fraud Intelligence
### What does a limited investigation queue catch?

**Exploratory case study · Python / pandas · Simulated card transactions**

**Business question.** If an investigator can review only a fixed number of transactions per day, how much fraud does a largest-amount-first queue capture, and how much legitimate activity enters that queue?

**Executive preview.** We verify the source, establish the prevalence of fraud labels, then compare two illustrative daily budgets. We repeat the comparison below the simulator's high-amount boundary to see how much the ranking depends on that rule.

**Evidence scope.** This is a retrospective description of the full simulated snapshot. A later test period can be excluded from fitting and model selection, but its aggregate data is already part of this exploration. Treat that evaluation as retrospective too.

| Section | Decision it informs |
|---|---|
| 1 · Data provenance and audit | Can we trust the input and its grain? |
| 2 · Fraud prevalence over time | What workload and fraud base rate should we expect? |
| 3 · The daily review queue | What changes when daily capacity doubles? |
| 4 · The simulator's amount rule | Does the result depend on the high-amount group? |
| 5 · Investigation and reporting implications | What evidence should the next phase produce? |

## 1 · Data provenance and audit

**Question.** Does the input match the audited source, with one row per simulated transaction?

**Approach.** Load the combined local CSV generated by the repository converter. Check the nine source fields, unique transaction IDs, date coverage, missingness, label domain and totals. A SHA-256 fingerprint records the exact CSV used in this run. The converter verifies the raw archive against the pinned manifest.

**Run locally.** Open this notebook from the existing repo with the project's Python kernel. Generate the input with `Prepare Transactions.cmd` if needed. See the [notebook setup instructions](../data/README.md#run-the-exploratory-notebook) and [source audit](../docs/data_sources.md).

In [ ]:
from hashlib import file_digest
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import PercentFormatter, StrMethodFormatter

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "convert_transactions.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook from the Banking Fraud Intelligence checkout.")

CSV_PATH = REPO_ROOT / "data" / "processed" / "transactions.csv"
if not CSV_PATH.is_file():
    raise FileNotFoundError("Run Prepare Transactions.cmd to create data/processed/transactions.csv.")

plt.rcParams.update({
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.labelcolor": "#334155",
    "axes.titlecolor": "#0F172A",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "grid.color": "#E2E8F0",
    "xtick.color": "#475569",
    "ytick.color": "#475569",
})

In [ ]:
source_columns = [
    "TRANSACTION_ID", "TX_DATETIME", "CUSTOMER_ID", "TERMINAL_ID",
    "TX_AMOUNT", "TX_TIME_SECONDS", "TX_TIME_DAYS", "TX_FRAUD",
    "TX_FRAUD_SCENARIO",
]
with CSV_PATH.open("rb") as source:
    csv_sha256 = file_digest(source, "sha256").hexdigest()
transactions = pd.read_csv(
    CSV_PATH, parse_dates=["TX_DATETIME"], date_format="ISO8601",
    float_precision="round_trip",
)

assert transactions.columns.tolist() == source_columns
assert len(transactions) == 1_754_155
assert transactions.TRANSACTION_ID.is_unique
assert transactions.TX_DATETIME.is_monotonic_increasing
assert not transactions.isna().any().any()
assert set(transactions.TX_FRAUD.unique()) == {0, 1}
assert transactions.TX_FRAUD.sum() == 14_681
assert transactions.TX_AMOUNT.ge(0).all()

transactions["day"] = transactions.TX_DATETIME.dt.normalize()
expected_days = pd.date_range("2018-04-01", "2018-09-30")
assert pd.DatetimeIndex(transactions.day.unique()).equals(expected_days)

audit = {
    "Transactions": f"{len(transactions):,}",
    "Source fields": len(source_columns),
    "Observation period": f"{expected_days[0]:%d %b %Y} to {expected_days[-1]:%d %b %Y}",
    "Calendar days": len(expected_days),
    "Duplicate transaction IDs": f"{transactions.TRANSACTION_ID.duplicated().sum():,}",
    "Missing source values": f"{transactions[source_columns].isna().sum().sum():,}",
    "Zero-amount transactions": f"{transactions.TX_AMOUNT.eq(0).sum():,}",
    "CSV SHA-256": csv_sha256,
}
display(pd.Series(audit, name="Verified value").to_frame())

**Interpretation.** The data has continuous daily coverage and a unique transaction key. Equal timestamps are possible, so the later modelling stage will need an explicit rule for which records count as earlier history. The source includes a retrospective fraud label and a simulator scenario field. It has no label-confirmation time. Amounts are shown in the dataset's units.

## 2 · Fraud prevalence over time

**Question.** How much fraud-labelled activity is there relative to normal transaction traffic?

**Approach.** Count labelled fraud by month and divide by the total number of transactions in the same month. The dotted reference is the pooled transaction-level rate.

In [ ]:
fraud_count = int(transactions.TX_FRAUD.sum())
pooled_rate = fraud_count / len(transactions)
monthly = (
    transactions.groupby(transactions.TX_DATETIME.dt.to_period("M"))
    .TX_FRAUD.agg(Transactions="size", Fraud="sum")
)
monthly.index = pd.PeriodIndex(monthly.index, freq="M", name=monthly.index.name).strftime("%b %Y")
monthly["Fraud rate"] = monthly.Fraud / monthly.Transactions

display(Markdown(
    f"**Snapshot:** {fraud_count:,} fraud-labelled transactions out of "
    f"{len(transactions):,} (**{pooled_rate:.2%}**)."
))
def format_table(frame, formats):
    return frame.assign(**{
        column: frame[column].map(pattern.format)
        for column, pattern in formats.items()
    })

display(format_table(monthly, {
    "Transactions": "{:,}", "Fraud": "{:,}", "Fraud rate": "{:.2%}",
}))

figure, axis = plt.subplots(figsize=(8, 3.2), layout="constrained")
axis.plot(monthly.index, monthly["Fraud rate"], color="#137A73", marker="o", linewidth=2.4)
axis.axhline(pooled_rate, color="#64748B", linestyle="--", label=f"Full snapshot · {pooled_rate:.2%}")
axis.set(title="Fraud prevalence by month", ylabel="Share of transactions", ylim=(0, 0.010))
axis.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=1))
axis.grid(axis="y")
axis.legend(frameon=False, loc="lower right")
plt.show()

**Interpretation.** Fraud is a small share of activity. A high overall accuracy figure could coexist with an unhelpful investigation queue. The monthly rates describe this simulator and period. They give context for using queue precision and fraud recall in the investigation.

## 3 · The daily review queue

**Question.** What do we gain by raising the review budget from 50 to 100 transactions per day?

**Approach.** Within each calendar day, rank eligible transactions by amount from highest to lowest, then by transaction ID from smallest to largest for equal amounts. Select the first N rows per day. Precision divides fraud caught by reviews. Recall divides fraud caught by all fraud labels in the eligible population for the same period. Amount capture divides captured fraud-labelled value by all fraud-labelled value in that population.

The daily ranking describes a review batch formed from that day's transactions. A real-time alerting policy would require a different timing protocol.

In [ ]:
def review_summary(population, limits=(50, 100)):
    ranked = population.sort_values(
        ["day", "TX_AMOUNT", "TRANSACTION_ID"], ascending=[True, False, True]
    )
    total_fraud = int(ranked.TX_FRAUD.sum())
    total_fraud_value = ranked.loc[ranked.TX_FRAUD.eq(1), "TX_AMOUNT"].sum()
    daily_queues = ranked.groupby("day", sort=False)
    results = []
    for limit in limits:
        selected = daily_queues.head(limit)
        caught = selected.loc[selected.TX_FRAUD.eq(1)]
        results.append({
            "Daily limit": limit,
            "Reviews": len(selected),
            "Fraud caught": len(caught),
            "Legitimate reviews": len(selected) - len(caught),
            "Precision": len(caught) / len(selected),
            "Recall": len(caught) / total_fraud,
            "Fraud amount capture": caught.TX_AMOUNT.sum() / total_fraud_value,
        })
    return pd.DataFrame(results).set_index("Daily limit")

metric_formats = {
    "Reviews": "{:,}", "Fraud caught": "{:,}", "Legitimate reviews": "{:,}",
    "Precision": "{:.2%}", "Recall": "{:.2%}", "Fraud amount capture": "{:.2%}",
}
baseline = review_summary(transactions)
assert (baseline.Reviews == baseline.index * len(expected_days)).all()
display(format_table(baseline, metric_formats))

figure, axis = plt.subplots(figsize=(8, 3.2), layout="constrained")
positions = range(len(baseline))
axis.barh(positions, baseline["Fraud caught"], color="#137A73", label="Fraud caught")
axis.barh(
    positions, baseline["Legitimate reviews"], left=baseline["Fraud caught"],
    color="#CBD5E1", label="Legitimate reviews",
)
axis.set_yticks(list(positions), [f"{limit} reviews/day" for limit in baseline.index])
axis.invert_yaxis()
axis.set(title="How the daily budget changes the queue", xlabel="Total reviews across 183 days")
axis.set_xlim(0, baseline.Reviews.max() * 1.14)
axis.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
for position, reviews in zip(positions, baseline.Reviews):
    axis.annotate(f"{reviews:,}", (reviews, position), xytext=(5, 0),
                  textcoords="offset points", va="center", color="#334155")
axis.legend(frameon=False, loc="upper right")
plt.show()

extra = baseline.loc[100] - baseline.loc[50]
display(Markdown(
    f"**Interpretation.** Doubling capacity caught **{extra['Fraud caught']:,.0f} "
    f"additional fraud cases** and sent **{extra['Legitimate reviews']:,.0f} "
    "additional legitimate transactions** to review in this simulation."
))

## 4 · The simulator's amount rule

**Question.** How much does the amount rule contribute to the apparent performance of the queue?

**Approach.** The [simulator documentation](https://fraud-detection-handbook.github.io/fraud-detection-handbook/Chapter_3_GettingStarted/SimulatedDataset.html) describes a deterministic high-amount rule. Verify that boundary in the snapshot. Re-rank the eligible population at or below 220 and apply the same daily limits. Use that population's fraud labels and value as the denominators. Compare the two populations descriptively.

In [ ]:
boundary = 220
above_boundary = transactions.TX_AMOUNT.gt(boundary)
high_amounts = transactions.loc[above_boundary]
assert high_amounts.TX_FRAUD.eq(1).all()
restricted = transactions.loc[~above_boundary]
restricted_fraud = int(restricted.TX_FRAUD.sum())
sensitivity = review_summary(restricted)
assert (sensitivity.Reviews == baseline.Reviews).all()

display(Markdown(
    f"**Rule footprint.** All **{len(high_amounts):,}** transactions above "
    f"{boundary} carry a fraud label. They represent **{len(high_amounts) / fraud_count:.1%}** "
    f"of all fraud labels. The eligible population at or below {boundary} contains "
    f"**{restricted_fraud:,}** fraud labels."
))
display(format_table(sensitivity, metric_formats))
display(Markdown(
    f"**Interpretation.** At 50 reviews per day, the amount-based rule caught "
    f"**{sensitivity.loc[50, 'Recall']:.2%}** of eligible fraud in the restricted population. "
    "This is a reason to test whether earlier customer or terminal activity improves "
    "the ranking when the deterministic group is excluded."
))

## 5 · Investigation and reporting implications

**What we learned.** The marginal benefit of a larger amount-based queue is small relative to the additional review workload in this snapshot. The simulator's high-amount rule materially affects that result.

**What to test next.** Define strictly prior customer and terminal features and handle equal timestamps explicitly. Choose the model on chronological validation data. Then compare the frozen model and the amount baseline on a later period excluded from fitting and model selection. Use identical daily budgets, report both populations, and keep the scenario label out of predictors. Report fraud-labelled value as exposure under review.

**What this means for Power BI.** A capacity comparison and investigation queue are candidates for the later report. The evaluated results, the agreed publication scope, and owner approval will determine the dashboard design.

[Evaluation methodology](../docs/methodology.md) · [Current project handoff](../docs/SESSION_HANDOFF_CURRENT.md)